**1. Установка необходимых библиотек.**

---



In [ ]:
import nltk
from nltk.corpus import stopwords
import sklearn
import re
import pandas as pd
!pip install keybert
!pip install transformers

**2. Чтение файла и получение информации о нём.**

---



In [ ]:
df = pd.read_csv('/content/sample_data/processed_reviews.csv', index_col=0)
print(df.describe())
print(df.info())
print(df.head(3))

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
print(df['review'])

In [ ]:
# Валидация значений
# очещаем Nan значения из таблицы в поле 'review'
df = df.dropna(subset=['review'])
total_chars_no_spaces = df['review'].apply(lambda x: len(str(x).replace(' ', ''))).sum()
print(total_chars_no_spaces)

**3. Предобработка текста**

---



In [ ]:
# облако слов до очистки
import matplotlib.pyplot as plt
from PIL import Image
from wordcloud import WordCloud

merged_text = ' '.join(df['review'])
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(merged_text)
plt.imshow(wordcloud)


In [ ]:
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')

In [ ]:
#токенезируем и удаляем стоп-слова
import string

stop_words = set(stopwords.words('russian')) | {'достоинства', 'комментарий', 'недостатки'} - {'не', 'ни', 'нет'}
print(stop_words)
# Множество знаков препинания
punctuation = set(string.punctuation)

def tokenize_and_clean(review):
    if pd.isna(review):
        return []

    tokens = nltk.word_tokenize(review, language='russian')


    tokens_no_punct = [t for t in tokens if t not in punctuation]

    tokens_lower = [t.lower() for t in tokens_no_punct]

    tokens_filtered = [
        t for t in tokens_lower
        if t not in stop_words
    ]
    return tokens_filtered

tokenize_text = [tokenize_and_clean(sentence) for sentence in df['review']]

print(tokenize_text)
print(len(tokenize_text))

In [ ]:
# лемматизация
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
lemmatized_text = [[lemmatizer.lemmatize(word) for word in sentence] for sentence in tokenize_text]
print(lemmatized_text)

# обединяем токены в предложения
final_text = [" ".join(sentence) for sentence in lemmatized_text]
print(final_text)

In [ ]:
total_chars_clean = sum(len(re.findall(r'\w', text)) for text in final_text)
print(total_chars_clean)

In [ ]:
# облако слов после очистки
merged_text = ' '.join(final_text)
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(merged_text)
plt.imshow(wordcloud)

**4. Определение ключевых слов и их значения важности**

---



In [ ]:
from keybert import KeyBERT

# Загружаем модель BERT
model = KeyBERT('DeepPavlov/rubert-base-cased')

# Выводим ключевые слова и коэффициент уверенности модели
keywords = [model.extract_keywords(sentence) for sentence in final_text]
print(keywords)

In [ ]:
from collections import defaultdict

word_scores = defaultdict(float)

for review in keywords:
    for word, score in review:
        word_scores[word] += score

top_n = 10

top_words = sorted(
    word_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:top_n]

words, scores = zip(*top_words)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.bar(words, scores)

plt.xticks(rotation=45)
plt.title("Топ ключевых слов")
plt.xlabel("Слова")
plt.ylabel("Суммарный вес")

plt.tight_layout()
plt.show()

In [ ]:
df['keywords'] = keywords
df

**5. Определение тональности отзывов.**

---



In [ ]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="cointegrated/rubert-tiny-sentiment-balanced"
)

def predict(text):
    result = classifier(text)[0]
    return result['label']

df['sentiment'] = df['review'].apply(predict)

In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt

counts = df['sentiment'].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(counts.index, counts.values)

plt.title('Распределение тональностей отзывов')
plt.xlabel('Класс')
plt.ylabel('Количество')

plt.show()

**6. Сохранение таблицы в виде csv-файла**

---



In [ ]:
df.to_csv('markets_reviews.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt

order = ['negative', 'neutral', 'positive']
counts = df['sentiment'].value_counts().reindex(order).fillna(0)

def autopct_format(values):
    def inner(pct):
        total = sum(values)
        val = int(round(pct * total / 100.0))
        return f'{val}\n({pct:.1f}%)'
    return inner

plt.figure(figsize=(6, 6))

plt.pie(
    counts.values,
    labels=counts.index,
    autopct=autopct_format(counts.values),
    startangle=90,
    colors = ['#f5b7b1', '#d5d8dc', '#a9dfbf']
)

plt.title('Распределение тональностей отзывов среди всех потребителей')

plt.show()

In [ ]:
counts = df.groupby(['market', 'sentiment']).size().unstack(fill_value=0)
counts[['positive', 'neutral', 'negative']].plot(kind='bar', figsize=(8, 5))

plt.title('Тональность отзывов по маркетплейсам')
plt.xlabel('Маркетплейс')
plt.ylabel('Количество отзывов')
plt.xticks(rotation=0)
plt.legend(title='Тональность')
plt.tight_layout()
plt.show()

In [ ]:
gender_counts = df['sex'].value_counts()
print(gender_counts)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

def extract_words(s):
    if pd.isna(s):
        return []
    return re.findall(r"'([^']+)'", str(s))

df = pd.read_csv('markets_reviews.csv')
df = df[df['sentiment'].isin(['positive', 'negative'])].copy()
df['words'] = df['keywords'].apply(extract_words)

rows = []
for _, row in df.iterrows():
    for w in row['words']:
        rows.append((row['sentiment'], w.lower().strip()))

kw_df = pd.DataFrame(rows, columns=['sentiment', 'word'])

plot_df = pd.DataFrame({
    'positive': kw_df[kw_df['sentiment'] == 'positive']['word'].value_counts().head(10),
    'negative': kw_df[kw_df['sentiment'] == 'negative']['word'].value_counts().head(10)
}).fillna(0)

plot_df.sort_values('positive').plot(kind='barh', figsize=(10, 6))
plt.title('Топ ключевых слов в положительных и негативных отзывах')
plt.xlabel('Частота')
plt.ylabel('Ключевое слово')
plt.tight_layout()
plt.show()

In [ ]:
counts_ = df.groupby(['sex', 'sentiment']).size().unstack(fill_value=0)
counts_[['positive', 'negative']].plot(kind='bar', figsize=(8, 5))

plt.title('Тональность отзывов относительно пола потребителей')
plt.xlabel('Пол')
plt.ylabel('Количество отзывов')
plt.xticks(rotation=0)
plt.legend(title='Тональность')
plt.tight_layout()
plt.show()